In [ ]:
import torch
from torchvision.models.detection import maskrcnn_resnet50_fpn

model = maskrcnn_resnet50_fpn(pretrained=False)
checkpoint = torch.load("outputs/maskrcnn_resnet50_fpn_20250609_1525/model_best.pth")
model.load_state_dict(checkpoint['model_state_dict'])  # Adjust key if needed
model.eval().cuda()

In [ ]:
import torch.nn.functional as F

def compute_entropy(prob_map):
    # prob_map: (C, H, W)
    entropy = -torch.sum(prob_map * torch.log(prob_map + 1e-8), dim=0)
    return entropy

def split_regions(prob_map, threshold=0.5):
    entropy = compute_entropy(prob_map)
    trusted_mask = (entropy < threshold).float()
    untrusted_mask = 1.0 - trusted_mask
    return trusted_mask, untrusted_mask

import torch.nn as nn

class CRA_Discriminator(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 1, 1)
        )

    def forward(self, x):
        return self.net(x)
